# ส่วนที่ 1: Clean ข้อมูล

โหลดข้อมูลดิบ ตรวจสอบคุณภาพ แล้วรวมรายชั่วโมงเป็นรายวัน บันทึกเป็น `daily_clean.csv` เพื่อนำไปใช้ในไฟล์วิเคราะห์

In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. โหลดและตรวจสอบข้อมูลดิบ

In [2]:
import pandas as pd
df = pd.read_csv(r'd:\python\ERA5_orchard_12.6510N_102.2128E_2020-2026_hourly.csv')
# 1. ตรวจสอบขนาดข้อมูล
print("ขนาดข้อมูล:", df.shape)

# 2. ตรวจสอบชื่อคอลัมน์
print("\nชื่อคอลัมน์:")
print(df.columns.tolist())

# 3. ตรวจสอบชนิดข้อมูล
print("\nชนิดข้อมูล:")
print(df.dtypes)

# 4. ตรวจสอบค่าว่าง
print("\nจำนวนค่าว่าง:")
print(df.isnull().sum())

# 5. ตรวจสอบข้อมูลซ้ำ
print("\nจำนวนแถวซ้ำ:", df.duplicated().sum())

# 6. แปลงเวลาและตรวจสอบช่วงวันที่
df["time_local"] = pd.to_datetime(df["time_local"], errors="coerce")

print("\nวันแรก:", df["time_local"].min())
print("วันสุดท้าย:", df["time_local"].max())
print("เวลาที่แปลงไม่ได้:", df["time_local"].isna().sum())

# 8. ตรวจสอบข้อมูลสถิติเบื้องต้น
print("\nสถิติเบื้องต้น:")
print(df.describe().round(3))

ขนาดข้อมูล: (59016, 25)

ชื่อคอลัมน์:
['time_local', 'temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'vapour_pressure_deficit', 'soil_temperature_0_to_7cm', 'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'soil_temperature_100_to_255cm', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm', 'soil_moisture_28_to_100cm', 'soil_moisture_100_to_255cm', 'precipitation', 'surface_pressure', 'cloud_cover', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'wind_speed_10m', 'wind_direction_10m', 'et0_fao_evapotranspiration', 'u10_ms', 'v10_ms', 'PAR_MJ_m2_h']

ชนิดข้อมูล:
time_local                           str
temperature_2m                   float64
dew_point_2m                     float64
relative_humidity_2m               int64
vapour_pressure_deficit          float64
soil_temperature_0_to_7cm        float64
soil_temperature_7_to_28cm       float64
soil_temperature_28_to_100cm     float64
soil_temperature_100_to_255cm    float64
soil_moisture_0_to_7cm 

## 3. รวมข้อมูลรายชั่วโมงเป็นรายวัน

In [3]:
# ================= A) รายชั่วโมง -> รายวัน =================
if "time_local" in df.columns:
    df["time_local"] = pd.to_datetime(df["time_local"])
    df = df.set_index("time_local").sort_index()

def d_sum(s):   # ทั้งวันไม่มีข้อมูล -> NaN (ไม่ใช่ 0)
    return s.resample("D").sum(min_count=1)

daily = pd.DataFrame({
    "rain":  d_sum(df["precipitation"]),
    "et0":   d_sum(df["et0_fao_evapotranspiration"]),
    "tmean": df["temperature_2m"].resample("D").mean(),
    "tmax":  df["temperature_2m"].resample("D").max(),
    "tmin":  df["temperature_2m"].resample("D").min(),
    "vpd":   df["vapour_pressure_deficit"].resample("D").mean(),
    "rh":    df["relative_humidity_2m"].resample("D").mean(),
    "rad":   df["shortwave_radiation"].resample("D").mean(),
    "cloud": df["cloud_cover"].resample("D").mean(),
    "wind":  df["wind_speed_10m"].resample("D").mean(),
})

# ความชื้นดิน: เฉลี่ย 2 ชั้น (0-7 และ 7-28 ซม.)
soil_layers = (df["soil_moisture_0_to_7cm"] + df["soil_moisture_7_to_28cm"]) / 2
daily["soil"] = soil_layers.resample("D").mean()

# จำนวนชั่วโมง/วันที่ VPD เกิน 2 kPa
v = df["vapour_pressure_deficit"]
daily["vpd_hours_high"] = (v > 2.0).astype(float).where(v.notna()).resample("D").sum(min_count=1)

# บันทึกตารางรายวันที่ clean แล้ว ให้ไฟล์วิเคราะห์โหลดต่อ
daily.to_csv(r'd:\python\daily_clean.csv')
print("บันทึกแล้ว:", daily.shape)


บันทึกแล้ว: (2459, 12)
